# ⚡ STUDIO VIDEO AI — MOTOR VIGGLE-ANIMATE (COMFYUI)
**Ponte de Alta Performance com GPU NVIDIA A100 / L4 (Google Colab)**

Este notebook inicializa o motor neural **Viggle-Animate (MiniMax-H3)** via ComfyUI integrado ao painel web **Studio Video AI**.
Ele executa:
1. Diagnóstico de GPU e trava de segurança para VRAM >= 22 GB;
2. Instalação do ComfyUI e dos nós customizados (`ComfyUI-Viggle-Animate-H3`, `ComfyUI-VideoHelperSuite`, `ComfyUI-KJNodes`, etc.);
3. Download automatizado dos pesos int8 e VAE via Hugging Face Hub;
4. Inicialização do ComfyUI local (`127.0.0.1:8188`) sem `--lowvram`;
5. Inicialização da API REST `colab_viggle_api.py` na porta `7860`;
6. Túnel público Cloudflare seguro (`trycloudflare.com`) para conectar ao site na Vercel.


## 1. Diagnóstico de Hardware (GPU e VRAM)
Verifica a presença da GPU e confirma se há pelo menos 22 GB de VRAM (ex.: A100 ou L4).


In [ ]:
# ==============================================================================
# 1. DIAGNÓSTICO DE HARDWARE: GPU E VRAM
# ==============================================================================
import sys
import torch

print("🔍 Verificando GPU disponível no ambiente...")

if not torch.cuda.is_available():
    print("\n" + "="*70)
    print("❌ ERRO: Nenhuma GPU detectada!")
    print("No menu do Colab, vá em: Ambiente de execução -> Alterar tipo de ambiente de execução")
    print("e selecione uma GPU (L4 ou A100 no Colab Pro).")
    print("="*70 + "\n")
    raise RuntimeError("Nenhuma GPU ativa no Google Colab.")

gpu_name = torch.cuda.get_device_name(0)
total_vram_bytes = torch.cuda.get_device_properties(0).total_memory
total_vram_gb = total_vram_bytes / (1024 ** 3)

print("\n" + "="*70)
print(f"🎮 GPU Detectada : {gpu_name}")
print(f"💾 VRAM Total    : {total_vram_gb:.2f} GB")
print("="*70 + "\n")

# Exibe tabela do nvidia-smi
!nvidia-smi

# Verificação de VRAM mínima de segurança (22 GB)
if total_vram_gb < 21.5:
    print("\n" + "!"*70)
    print("🛑 BLOQUEIO DE CAPACIDADE DE MEMÓRIA VRAM:")
    print("O menor modelo tem 21 GB; use L4 ou A100 (Colab Pro)")
    print("!"*70 + "\n")
    raise RuntimeError("O menor modelo tem 21 GB; use L4 ou A100 (Colab Pro)")

print("\n✅ VRAM suficiente detectada! Você pode prosseguir para a próxima célula.")


## 2. Instalação do ComfyUI, Custom Nodes e Dependências
Instala o ComfyUI oficial, os nós do Viggle-Animate, VideoHelperSuite, KJNodes, ferramentas de mídia (ffmpeg, edge-tts, opencv) e dependências.


In [ ]:
# ==============================================================================
# 2. INSTALAÇÃO DO COMFYUI E CUSTOM NODES
# ==============================================================================
import os
import subprocess

print("📦 1/3 Instalando ComfyUI oficial...")
%cd /content
if not os.path.exists("/content/ComfyUI"):
    !git clone https://github.com/comfyanonymous/ComfyUI.git

%cd /content/ComfyUI
!pip install -q -r requirements.txt
!pip install -q huggingface_hub flask flask-cors edge-tts opencv-python pillow

print("\n🧩 2/3 Clonando e configurando Custom Nodes...")
custom_nodes_dir = "/content/ComfyUI/custom_nodes"
os.makedirs(custom_nodes_dir, exist_ok=True)
%cd {custom_nodes_dir}

# 1. ComfyUI-Viggle-Animate-H3 (Saganaki22)
if not os.path.exists("ComfyUI-Viggle-Animate-H3"):
    !git clone https://github.com/Saganaki22/ComfyUI-Viggle-Animate-H3.git
if os.path.exists("ComfyUI-Viggle-Animate-H3/requirements.txt"):
    !pip install -q -r ComfyUI-Viggle-Animate-H3/requirements.txt

# 2. ComfyUI-VideoHelperSuite (VHS - Kosinkadink)
if not os.path.exists("ComfyUI-VideoHelperSuite"):
    !git clone https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git
if os.path.exists("ComfyUI-VideoHelperSuite/requirements.txt"):
    !pip install -q -r ComfyUI-VideoHelperSuite/requirements.txt

# 3. ComfyUI-KJNodes (necessário para os nós do workflow: CustomSigmas, ModelPreviewOverrideKJ)
if not os.path.exists("ComfyUI-KJNodes"):
    !git clone https://github.com/kijai/ComfyUI-KJNodes.git
if os.path.exists("ComfyUI-KJNodes/requirements.txt"):
    !pip install -q -r ComfyUI-KJNodes/requirements.txt

# 4. comfyui-various (para o nó JWImageResizeToMegapixels presente no workflow)
if not os.path.exists("comfyui-various"):
    !git clone https://github.com/jamesWalker55/comfyui-various.git

%cd /content
print("\n✅ ComfyUI e nós customizados instalados com sucesso!")


## 3. Download dos Modelos do Viggle-Animate
Baixa os pesos oficiais diretamente do Hugging Face Hub para as pastas corretas de modelos do ComfyUI, pulando arquivos já baixados.


In [ ]:
# ==============================================================================
# 3. DOWNLOAD DOS MODELOS VIGGLE-ANIMATE (HUGGING FACE)
# ==============================================================================
import os
import shutil
from huggingface_hub import HfApi, hf_hub_download

api = HfApi()

DOWNLOAD_LIST = [
    {
        "name": "Diffusion Model (int8 pruned, ~21 GB)",
        "repo_id": "drbaph/Viggle-Animate-ComfyUI",
        "repo_file": "diffusion_models/minimax_h3_ref2va_viggle_pruned_int8_convrot.safetensors",
        "target_dir": "/content/ComfyUI/models/diffusion_models",
        "target_file": "minimax_h3_ref2va_viggle_pruned_int8_convrot.safetensors"
    },
    {
        "name": "LoRA DMD 4-Step (r64, ~0.94 GB)",
        "repo_id": "drbaph/Viggle-Animate-ComfyUI",
        "repo_file": "loras/viggle_animate_dmd_lora_r64.safetensors",
        "target_dir": "/content/ComfyUI/models/loras",
        "target_file": "viggle_animate_dmd_lora_r64.safetensors"
    },
    {
        "name": "Text Conditioning Embeddings (~1.5 MB)",
        "repo_id": "drbaph/Viggle-Animate-ComfyUI",
        "repo_file": "text_cond/fixed_embed_fwd_anyframe.safetensors",
        "target_dir": "/content/ComfyUI/models/text_cond",
        "target_file": "fixed_embed_fwd_anyframe.safetensors"
    },
    {
        "name": "Video VAE int8 convrot (~3.17 GB)",
        "repo_id": "Kijai/MiniMax-H3-experimental",
        "repo_file": "minimax_h3_video_vae_int8_convrot.safetensors",
        "target_dir": "/content/ComfyUI/models/vae",
        "target_file": "minimax_h3_video_vae_int8_convrot.safetensors"
    }
]

print("🚀 Verificando e baixando modelos necessários...\n")

for item in DOWNLOAD_LIST:
    repo_id = item["repo_id"]
    repo_file = item["repo_file"]
    target_dir = item["target_dir"]
    target_file = item["target_file"]
    dest_path = os.path.join(target_dir, target_file)

    os.makedirs(target_dir, exist_ok=True)

    if os.path.exists(dest_path) and os.path.getsize(dest_path) > 1024:
        size_mb = os.path.getsize(dest_path) / (1024 * 1024)
        print(f"⏩ [PULADO] Já existe: {dest_path} ({size_mb:.2f} MB)")
        continue

    print(f"📡 Checando arquivo '{repo_file}' em '{repo_id}'...")
    repo_files = api.list_repo_files(repo_id)
    if repo_file not in repo_files:
        raise FileNotFoundError(f"Arquivo '{repo_file}' inexistente em '{repo_id}'.")

    print(f"⬇️ Baixando {item['name']}...")
    downloaded_path = hf_hub_download(
        repo_id=repo_id,
        filename=repo_file,
        local_dir="/content/hf_cache",
        local_dir_use_symlinks=False
    )

    if os.path.abspath(downloaded_path) != os.path.abspath(dest_path):
        shutil.move(downloaded_path, dest_path)

    print(f"✅ Concluído: {dest_path} ({os.path.getsize(dest_path)/(1024*1024):.2f} MB)\n")

# Baixar o workflow API base do repositório GitHub
os.makedirs("/content/workflows", exist_ok=True)
wf_dest = "/content/workflows/viggle_api.json"
if not os.path.exists(wf_dest) or os.path.getsize(wf_dest) < 100:
    import urllib.request
    import time
    wf_url = f"https://raw.githubusercontent.com/grego921/studio-video-ai/viggle-integracao/workflows/viggle_api.json?t={int(time.time())}"
    print("📥 Baixando workflow base viggle_api.json do GitHub...")
    urllib.request.urlretrieve(wf_url, wf_dest)

print("🎉 Todos os modelos e workflows estão prontos!")


## 4. Iniciar o ComfyUI (Modo Alta Performance)
Inicia o ComfyUI em segundo plano sem `--lowvram` (aproveitando os 40 GB/80 GB da A100 para máxima velocidade).


In [ ]:
# ==============================================================================
# 4. INICIALIZAÇÃO DO COMFYUI (SEM --lowvram)
# ==============================================================================
import os
import time
import subprocess
import urllib.request

!pkill -f "python main.py" || true
time.sleep(1)

print("🚀 Iniciando ComfyUI oficial na porta 8188 (sem --lowvram)...")
comfy_log = open('/content/comfyui.log', 'w')
comfy_proc = subprocess.Popen(
    ['python', '/content/ComfyUI/main.py', '--listen', '127.0.0.1', '--port', '8188'],
    stdout=comfy_log,
    stderr=subprocess.STDOUT
)

print("⏳ Aguardando ComfyUI responder na porta 8188...")
comfy_ready = False
for _ in range(40):
    try:
        with urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=1.5) as resp:
            if resp.status == 200:
                comfy_ready = True
                break
    except Exception:
        pass
    time.sleep(1)

if comfy_ready:
    print("✅ ComfyUI ativo e pronto para receber tarefas!")
else:
    print("⚠️ ComfyUI ainda inicializando nós; prosseguindo para subir a ponte de API...")


## 5. Iniciar API REST e Túnel Cloudflared
Baixa a versão mais recente de `colab_viggle_api.py` do GitHub raw (com cache-buster), inicia o servidor Flask na porta `7860` e expõe via túnel público seguro do Cloudflare.


In [ ]:
# ==============================================================================
# 5. INICIAR SERVIDOR API E TÚNEL CLOUDFLARE (PORTA 7860)
# ==============================================================================
import os
import re
import time
import subprocess
import urllib.request

# 1. Instalar Cloudflared se necessário
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

# 2. Atualizar colab_viggle_api.py do GitHub raw com cache-buster
print("🔄 Baixando colab_viggle_api.py do GitHub (branch viggle-integracao)...")
api_url = f"https://raw.githubusercontent.com/grego921/studio-video-ai/viggle-integracao/colab_viggle_api.py?t={int(time.time())}"
urllib.request.urlretrieve(api_url, "/content/colab_viggle_api.py")

# 3. Encerrar processos anteriores
!pkill -f cloudflared || true
!pkill -f colab_viggle_api.py || true
time.sleep(1)

# 4. Iniciar colab_viggle_api.py na porta 7860
print("🚀 Iniciando colab_viggle_api.py na porta 7860...")
api_log = open('/content/viggle_api.log', 'w')
env = os.environ.copy()
env["PORT"] = "7860"
env["COMFYUI_DIR"] = "/content/ComfyUI"
env["WORKFLOW_API_PATH"] = "/content/workflows/viggle_api.json"

api_proc = subprocess.Popen(
    ['python', '-u', '/content/colab_viggle_api.py'],
    stdout=api_log,
    stderr=subprocess.STDOUT,
    env=env
)

time.sleep(2)

# 5. Iniciar Túnel Cloudflared apontando para a porta 7860
print("🌐 Abrindo túnel público Cloudflare...")
tunnel_log = open('/content/tunnel.log', 'w')
tunnel_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:7860'],
    stdout=tunnel_log,
    stderr=subprocess.STDOUT
)

tunnel_url = None
for _ in range(40):
    time.sleep(1)
    if os.path.exists('/content/tunnel.log'):
        with open('/content/tunnel.log', 'r') as f:
            content = f.read()
            match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', content)
            if match:
                tunnel_url = match.group(0)
                break

if tunnel_url:
    print("\n" + "="*75)
    print("🎉 MOTOR GPU VIGGLE-ANIMATE CONECTADO COM SUCESSO!")
    print(f"👉 URL DO MOTOR GPU : {tunnel_url}")
    print("="*75 + "\n")
    print("1. Copie a URL acima;")
    print("2. No site (painel web), clique no botão 'Conectar Motor GPU';")
    print("3. Cole a URL e clique em 'Salvar e Conectar';")
    print("4. Mantenha esta célula executando durante o uso da ferramenta.")
else:
    print("❌ Falha ao obter a URL pública do túnel. Verifique /content/tunnel.log")
